### 이미지 -> mp4변경

In [ ]:
import os
import re
import cv2
import json
import random
from pathlib import Path
from collections import defaultdict

from ucf_utils import (
    collect_frames_by_video,
    frames_to_video,
    timestamps_to_output_frame_intervals,
)

# =========================
# 경로 설정
# =========================

# UCF-Crime 이미지 프레임 루트
# 예:
# C:/UCF-Crime/
# ├── Abuse/
# │   ├── Abuse001_x264_0.png
# │   ├── Abuse001_x264_10.png
# │   └── ...
# ├── Assault/
# └── Normal/
FRAME_ROOT = Path("C:/4_1_딥러닝_팀플/UCF-Crime")

# json annotation 폴더
# 예:
# C:/4_1_딥러닝_팀플/UCF-Crime-Annotations/
# ├── Abuse.json
# ├── Assault.json
# └── ...
ANNOTATION_TXT = Path("C:/4_1_딥러닝_팀플/UCF-Crime-Annotations/Temporal_Anomaly_Annotation.txt")

# MTFL용 출력 폴더
OUT_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom")

VIDEO_OUT = OUT_ROOT / "videos"
ANNO_OUT = OUT_ROOT / "annotations"

VIDEO_OUT.mkdir(parents=True, exist_ok=True)
ANNO_OUT.mkdir(parents=True, exist_ok=True)

# 원본 UCF-Crime frame이 보통 30fps 기준이고,
# 이미지 파일명이 _0, _10, _20처럼 10프레임 간격이면 frame_step=10
ORIGINAL_FPS = 30
DEFAULT_FRAME_STEP = 10

# 변환된 mp4 fps
# 10프레임마다 하나씩 저장된 이미지라면 30/10 = 3fps로 저장해야 원래 시간 길이와 비슷해짐
OUTPUT_FPS = ORIGINAL_FPS / DEFAULT_FRAME_STEP

random.seed(42)

ModuleNotFoundError: No module named 'ucf_utils'

### 전체 변환

In [ ]:
all_items = []

# =========================
# txt annotation 로드
# =========================
anno_map = {}

with open(ANNOTATION_TXT, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue

        parts = line.split()
        video_name = parts[0]          # Abuse028_x264.mp4
        category = parts[1]            # Abuse / Normal
        nums = list(map(int, parts[2:]))

        # 확장자 제거: Abuse028_x264.mp4 -> Abuse028_x264
        video_id = Path(video_name).stem

        intervals_raw = []
        for i in range(0, len(nums), 2):
            s, e = nums[i], nums[i + 1]
            if s != -1 and e != -1:
                intervals_raw.append((s, e))

        anno_map[video_id] = {
            "category": category,
            "intervals_raw": intervals_raw
        }

print("Loaded txt annotations:", len(anno_map))


# =========================
# frame 폴더 순회
# =========================
for category_dir in FRAME_ROOT.iterdir():
    if not category_dir.is_dir():
        continue

    category = category_dir.name
    print(f"\nProcessing category: {category}")

    grouped = collect_frames_by_video(category_dir)
    print(f"  videos found: {len(grouped)}")

    for video_id, frame_items in grouped.items():
        if len(frame_items) < 8:
            print("  skip too short:", video_id, len(frame_items))
            continue

        out_video_path = VIDEO_OUT / category / f"{video_id}.mp4"

        # 이미지 프레임 -> mp4
        if not out_video_path.exists():
            frames_to_video(frame_items, out_video_path, OUTPUT_FPS)

        total_output_frames = len(frame_items)

        # txt annotation에서 anomaly 구간 가져오기
        raw_intervals = anno_map.get(video_id, {}).get("intervals_raw", [])

        intervals = []
        for s, e in raw_intervals:
            # txt의 s,e는 원본 30fps 기준 frame index
            # 현재 mp4는 10프레임마다 1장으로 만들었으므로 / DEFAULT_FRAME_STEP
            out_s = int(s / DEFAULT_FRAME_STEP)
            out_e = int(e / DEFAULT_FRAME_STEP)

            out_s = max(0, min(out_s, total_output_frames - 1))
            out_e = max(0, min(out_e, total_output_frames - 1))

            if out_e > out_s:
                intervals.append((out_s, out_e))

        rel_video = f"{category}/{video_id}.mp4"

        # label 결정
        if video_id in anno_map:
            label = anno_map[video_id]["category"]
        else:
            label = "Normal" if category.lower() == "normal" else category

        all_items.append({
            "rel_video": rel_video,
            "label": label,
            "total_frames": total_output_frames,
            "intervals": intervals
        })

print("\nTotal converted videos:", len(all_items))

In [ ]:
import random
from pathlib import Path

random.seed(42)
random.shuffle(all_items)

test_ratio = 0.2
n_test = int(len(all_items) * test_ratio)

test_items = all_items[:n_test]
train_items = all_items[n_test:]

train_anno_path = ANNO_OUT / "train_anno.txt"
test_anno_path = ANNO_OUT / "test_anno.txt"

with open(train_anno_path, "w", encoding="utf-8") as f:
    for item in train_items:
        f.write(f"{item['rel_video']} {item['label']}\n")

with open(test_anno_path, "w", encoding="utf-8") as f:
    for item in test_items:
        line = f"{item['rel_video']} {item['label']} {item['total_frames']}"

        for s, e in item["intervals"]:
            line += f" {s} {e}"

        f.write(line + "\n")

print("train:", len(train_items), train_anno_path)
print("test:", len(test_items), test_anno_path)

In [ ]:
print("===== train sample =====")
print("\n".join(train_anno_path.read_text(encoding="utf-8").splitlines()[:10]))

print("\n===== test sample =====")
print("\n".join(test_anno_path.read_text(encoding="utf-8").splitlines()[:10]))

### train/test annotation 생성

In [ ]:
# train/test split
random.shuffle(all_items)

test_ratio = 0.2
n_test = int(len(all_items) * test_ratio)

test_items = all_items[:n_test]
train_items = all_items[n_test:]

train_anno_path = ANNO_OUT / "train_anno.txt"
test_anno_path = ANNO_OUT / "test_anno.txt"

with open(train_anno_path, "w", encoding="utf-8") as f:
    for item in train_items:
        f.write(f"{item['rel_video']} {item['label']}\n")

with open(test_anno_path, "w", encoding="utf-8") as f:
    for item in test_items:
        line = f"{item['rel_video']} {item['label']} {item['total_frames']}"

        for s, e in item["intervals"]:
            line += f" {s} {e}"

        f.write(line + "\n")

print("train:", len(train_items), train_anno_path)
print("test:", len(test_items), test_anno_path)

### annotation 확인

In [ ]:
print("===== train_anno sample =====")
print(train_anno_path.read_text(encoding="utf-8").splitlines()[:5])

print("\n===== test_anno sample =====")
print(test_anno_path.read_text(encoding="utf-8").splitlines()[:5])

In [ ]:
from collections import Counter

def print_stats(items, name):
    labels = [x["label"] for x in items]

    total = len(labels)

    normal_cnt = sum(1 for l in labels if l.lower() == "normal")
    abnormal_cnt = total - normal_cnt

    print(f"\n===== {name} =====")
    print(f"total videos     : {total}")
    print(f"normal videos    : {normal_cnt}")
    print(f"abnormal videos  : {abnormal_cnt}")

    print(f"normal ratio     : {normal_cnt / total:.4f}")
    print(f"abnormal ratio   : {abnormal_cnt / total:.4f}")

    print("\nlabel distribution:")
    counter = Counter(labels)

    for k, v in sorted(counter.items()):
        print(f"{k:15s}: {v}")

print_stats(train_items, "TRAIN")
print_stats(test_items, "TEST")

In [ ]:
import shutil

ffmpeg_path = shutil.which("ffmpeg")

print(ffmpeg_path)

## MTFL feature extraction

In [ ]:
import subprocess
from pathlib import Path

####
ffmpeg_path = Path(r"C:\Users\dongwon\anaconda3\pkgs\ffmpeg-8.1.1-gpl_h7d7abef_901\Library\bin\ffmpeg.exe")
####

result = subprocess.run(
    [str(ffmpeg_path), "-hide_banner", "-encoders"],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print("return:", result.returncode)
print("libx264 있음?:", "libx264" in result.stdout)
print(result.stderr[-1000:])

## 파일경로 변경 필요

In [ ]:
import sys
import subprocess
from pathlib import Path
import os
#####
os.chdir(r"C:\4_1_딥러닝_팀플\MTFL")
#####
MTFL_ROOT = "C:/4_1_딥러닝_팀플/MTFL" # MTFL 위치
CUSTOM_ROOT = "C:/4_1_딥러닝_팀플/MTFL_UCF_custom" ### 데이터 셋 위치 -> ucf crime dataset위치로

feature_script = f"{MTFL_ROOT}/utils/feature_extractor.py"
video_dir = f"{CUSTOM_ROOT}/videos"
feature_dir = f"{CUSTOM_ROOT}/features"
weight_path = "C:/4_1_딥러닝_팀플/MTFL_custom/swin_base_patch244_window877_kinetics400_22k.pth"

Path(feature_dir).mkdir(parents=True, exist_ok=True)

for clip_length in [8, 32, 64]:
    print(f"\n========== Extracting L{clip_length} ==========")

    cmd = [
        sys.executable,
        feature_script,
        "--clip_length", str(clip_length),
        "--dataset_path", video_dir,
        "--save_dir", feature_dir,
        "--pretrained_3d", weight_path,
        "--gpu", "0",
        "--batch_size", "4",
        "--num_workers", "4",
    ]

    result = subprocess.run(cmd, capture_output=True, text=True)

    print("RETURN CODE:", result.returncode)
    print(result.stdout)
    print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"L{clip_length} failed")

확인

In [ ]:
from pathlib import Path

for L in ["L8", "L32", "L64"]:
    files = list(Path(f"C:/4_1_딥러닝_팀플/MTFL_UCF_custom/features/{L}").rglob("*.txt"))
    print(L, len(files))

추출 안된 영상 확인

In [ ]:
from pathlib import Path

FEATURE_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom/features")

feature_sets = {}

for L in ["L8", "L32", "L64"]:
    files = list((FEATURE_ROOT / L).rglob("*.txt"))

    # L8/Shoplifting/Shoplifting007_x264.txt
    # -> Shoplifting/Shoplifting007_x264
    rels = {
        str(p.relative_to(FEATURE_ROOT / L).with_suffix("")).replace("\\", "/")
        for p in files
    }

    feature_sets[L] = rels
    print(L, len(rels))

common = feature_sets["L8"] & feature_sets["L32"] & feature_sets["L64"]

print("common:", len(common))
print("missing in L8:", len(common ^ feature_sets["L8"]))
print("missing in L32 compared to L8:", len(feature_sets["L8"] - feature_sets["L32"]))
print("missing in L64 compared to L8:", len(feature_sets["L8"] - feature_sets["L64"]))

print("\n=== L32에 없는 파일 예시 ===")
for x in sorted(feature_sets["L8"] - feature_sets["L32"])[:20]:
    print(x)

print("\n=== L64에 없는 파일 예시 ===")
for x in sorted(feature_sets["L8"] - feature_sets["L64"])[:20]:
    print(x)

In [ ]:
from pathlib import Path

CUSTOM_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom")
FEATURE_ROOT = CUSTOM_ROOT / "features"

def get_feature_set(L):
    files = list((FEATURE_ROOT / L).rglob("*.txt"))
    return {
        str(p.relative_to(FEATURE_ROOT / L).with_suffix("")).replace("\\", "/")
        for p in files
    }

sets = {L: get_feature_set(L) for L in ["L8", "L32", "L64"]}

missing_L32 = sorted(sets["L8"] - sets["L32"])
missing_L64 = sorted(sets["L8"] - sets["L64"])

print("missing L32:", len(missing_L32))
print("missing L64:", len(missing_L64))

miss_dir = CUSTOM_ROOT / "missing_lists"
miss_dir.mkdir(parents=True, exist_ok=True)

(miss_dir / "missing_L32.txt").write_text("\n".join(missing_L32), encoding="utf-8")
(miss_dir / "missing_L64.txt").write_text("\n".join(missing_L64), encoding="utf-8")

print("saved:", miss_dir / "missing_L32.txt")
print("saved:", miss_dir / "missing_L64.txt")

영상이 짧아서 추출이 안되는 거 -> 추출 안된거 제외하고 detection모델에 학습

L8, L32, L64의 집합

In [ ]:
from pathlib import Path

CUSTOM_ROOT = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom")
FEATURE_ROOT = CUSTOM_ROOT / "features"
ANNO_ROOT = CUSTOM_ROOT / "annotations"

def get_feature_set(L):
    return {
        str(p.relative_to(FEATURE_ROOT / L).with_suffix("")).replace("\\", "/")
        for p in (FEATURE_ROOT / L).rglob("*.txt")
    }

set_L8 = get_feature_set("L8")
set_L32 = get_feature_set("L32")
set_L64 = get_feature_set("L64")

common = set_L8 & set_L32 & set_L64

print("L8:", len(set_L8))
print("L32:", len(set_L32))
print("L64:", len(set_L64))
print("common:", len(common))

annotation도 수정

In [ ]:
def filter_annotation(input_path, output_path, common):
    kept = 0
    removed = 0

    with open(input_path, "r", encoding="utf-8") as fin, \
         open(output_path, "w", encoding="utf-8") as fout:

        for line in fin:
            if not line.strip():
                continue

            items = line.strip().split()
            video_path = items[0]

            # Shoplifting/Shoplifting007_x264.mp4
            # -> Shoplifting/Shoplifting007_x264
            rel = str(Path(video_path).with_suffix("")).replace("\\", "/")

            if rel in common:
                fout.write(line)
                kept += 1
            else:
                removed += 1

    print(output_path.name)
    print(" kept:", kept)
    print(" removed:", removed)


filter_annotation(
    ANNO_ROOT / "train_anno.txt",
    ANNO_ROOT / "train_anno_common.txt",
    common
)

filter_annotation(
    ANNO_ROOT / "test_anno.txt",
    ANNO_ROOT / "test_anno_common.txt",
    common
)

normal/anomaly 비율 확인

In [ ]:
from collections import Counter

def check_label_dist(path):
    labels = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                labels.append(line.split()[1])

    counter = Counter(labels)
    total = len(labels)
    normal = counter.get("Normal", 0)
    abnormal = total - normal

    print("\n", path.name)
    print("total:", total)
    print("normal:", normal)
    print("abnormal:", abnormal)
    print("normal ratio:", normal / total if total else 0)

    for k, v in sorted(counter.items()):
        print(k, v)

check_label_dist(ANNO_ROOT / "train_anno_common.txt")
check_label_dist(ANNO_ROOT / "test_anno_common.txt")

NameError: name 'ANNO_ROOT' is not defined

### MTFL detection train 실행

In [ ]:
import sys
import subprocess
from pathlib import Path

MTFL_ROOT = "C:/4_1_딥러닝_팀플/MTFL"
CUSTOM_ROOT = "C:/4_1_딥러닝_팀플/MTFL_UCF_custom"

Path(f"{CUSTOM_ROOT}/checkpoints_common").mkdir(parents=True, exist_ok=True)
Path(f"{CUSTOM_ROOT}/train_results_common").mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    f"{MTFL_ROOT}/detection/train.py",

    # 공통 feature만 있는 annotation 사용
    "--train_anno", f"{CUSTOM_ROOT}/annotations/train_anno_common.txt",
    "--test_anno", f"{CUSTOM_ROOT}/annotations/test_anno_common.txt",

    "--lf_dir", f"{CUSTOM_ROOT}/features/L64",
    "--mf_dir", f"{CUSTOM_ROOT}/features/L32",
    "--sf_dir", f"{CUSTOM_ROOT}/features/L8",

    "--save_models", f"{CUSTOM_ROOT}/checkpoints_common",
    "--output_dir", f"{CUSTOM_ROOT}/train_results_common",

    "--gpu", "0",
    "--feature_size", "1024",
    "--seg_num", "32",

    # detection 학습은 비교적 가벼움
    "--batch-size", "64",

    # Windows에서는 0이 안전
    "--workers", "0",

    "--lr", "0.0001",
    "--max-epoch", "2000",
]

result = subprocess.run(
    cmd,
    cwd=MTFL_ROOT,
)

print("RETURN CODE:", result.returncode)

학습 확인

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

RESULT_DIR = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom/train_results_common")

records = []

for txt_path in RESULT_DIR.glob("*-step-AUC.txt"):
    step_match = re.search(r"(\d+)-step-AUC\.txt", txt_path.name)
    if step_match is None:
        continue

    step = int(step_match.group(1))
    data = {"step": step}

    with open(txt_path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if ":" not in line:
                continue

            key, value = line.strip().split(":", 1)
            key = key.strip()
            value = value.strip()

            try:
                data[key] = float(value)
            except ValueError:
                pass

    records.append(data)

df = pd.DataFrame(records).sort_values("step")

print(df)

plt.figure(figsize=(8, 5))

if "AUC" in df.columns:
    plt.plot(df["step"], df["AUC"], marker="o", label="AUC")

if "AP" in df.columns:
    plt.plot(df["step"], df["AP"], marker="o", label="AP")

plt.xlabel("Step")
plt.ylabel("Score")
plt.title("MTFL Detection Evaluation Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

if "AUC" in df.columns:
    plt.plot(df["step"], df["AUC"], marker="o", label="AUC")

# if "AP" in df.columns:
#     plt.plot(df["step"], df["AP"], marker="o", label="AP")

plt.xlabel("Step")
plt.ylabel("Score")
plt.title("MTFL Detection Evaluation Curve")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from pathlib import Path

RESULT_DIR = Path("C:/4_1_딥러닝_팀플/MTFL_UCF_custom/train_results_common")

print("exists:", RESULT_DIR.exists())

for p in RESULT_DIR.iterdir():
    print(p.name)

### checkpoint 확인

In [ ]:
from pathlib import Path

ckpts = list(Path(f"{CUSTOM_ROOT}/checkpoints_common").rglob("*"))
for c in ckpts:
    print(c)

### 학습된 모델로 test

In [ ]:
import sys
import subprocess
from pathlib import Path

MTFL_ROOT = "C:/4_1_딥러닝_팀플/MTFL"
CUSTOM_ROOT = "C:/4_1_딥러닝_팀플/MTFL_UCF_custom"

detection_model = "C:/4_1_딥러닝_팀플/MTFL_UCF_custom/checkpoints_common/MTFL-1480.pkl"

Path(f"{CUSTOM_ROOT}/results_eval").mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable,
    f"{MTFL_ROOT}/detection/test.py",

    "--test_anno", f"{CUSTOM_ROOT}/annotations/test_anno_common.txt",
    "--detection_model", detection_model,

    "--lf_dir", f"{CUSTOM_ROOT}/features/L64",
    "--mf_dir", f"{CUSTOM_ROOT}/features/L32",
    "--sf_dir", f"{CUSTOM_ROOT}/features/L8",

    "--output_dir", f"{CUSTOM_ROOT}/results_eval",
    "--gpu", "0",
    "--feature_size", "1024",
    "--seg_num", "32",
    "--workers", "0",
]

result = subprocess.run(
    cmd,
    cwd=MTFL_ROOT,
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace"
)

print("RETURN CODE:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

CUSTOM_ROOT = Path(r"C:/4_1_딥러닝_팀플/MTFL_UCF_custom")
score_root = CUSTOM_ROOT / "results_eval" / "scores"

score_files = sorted(score_root.rglob("*_scores.npy"))

print("score files:", len(score_files))
for p in score_files[:10]:
    print(p)

In [ ]:
print("score file 개수:", len(score_files))

for i, p in enumerate(score_files[:20]):
    print(i, p.name)

### 시각화

In [ ]:
from MTFL_vis import (
    plot_saved_score,
    plot_score_with_selected_frames

)

In [ ]:
plot_saved_score(score_files[11])

In [ ]:
target_idx = 2

score_path = score_files[target_idx]

scores = np.load(score_path)

gt_path = score_path.with_name(
    score_path.name.replace("_scores.npy", "_gt.npy")
)

gt = np.load(gt_path)

anomaly_idx = np.where(gt == 1)[0]

print("video:", score_path.stem)
print("GT start:", anomaly_idx.min())
print("GT end:", anomaly_idx.max())

plot_score_with_selected_frames(
    score_path=score_path,
    video_root=r"C:\4_1_딥러닝_팀플\MTFL_UCF_custom\videos",
    frame_indices=sorted([
        int(anomaly_idx.min()),
        int(np.argmax(scores)),
        int(anomaly_idx.max())
    ])
)

In [ ]:
CUSTOM_ROOT = Path(r"C:\4_1_딥러닝_팀플\MTFL_custom_12_2")
AIHUB_SCORE_ROOT = CUSTOM_ROOT / "results_eval" / "scores"
details = event_hit_rate(score_root, top_percent=5)

for d in details[:10]:
    print(d)